In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into kernel variables (full dataframes kept in memory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to summarize a dataframe
def summarize_df(df, key_col):
    # total rows
    row_count = len(df)
    # timestamps -> timezone-aware in Asia/Shanghai
    if 'timestamp' in df.columns and row_count > 0:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = ts.min()
        max_ts = ts.max()
    else:
        min_ts = pd.NaT
        max_ts = pd.NaT
    # top unique values of key column (by frequency), limited to 20
    if key_col in df.columns:
        top_keys = df[key_col].value_counts().index[:20].tolist()
    else:
        top_keys = []
    # unique cmdb_id values (preserve appearance order), limit display to first 50
    if 'cmdb_id' in df.columns:
        cmdb_uniques = pd.unique(df['cmdb_id']).tolist()
        cmdb_display = cmdb_uniques[:50]
        cmdb_count = len(cmdb_uniques)
    else:
        cmdb_display = []
        cmdb_count = 0
    return {
        'rows': row_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        f'top_{key_col}_(<=20)': top_keys,
        'sample_cmdb_ids_(<=50)': cmdb_display,
        'unique_cmdb_count': cmdb_count
    }

# Build summaries for each file
summary_metric = summarize_df(df_metric, 'kpi_name')
summary_trace = summarize_df(df_trace, 'trace_name')
summary_log = summarize_df(df_log, 'log_name')
summary_error = summarize_df(df_error, 'message')

# Combine into a compact DataFrame for display
summaries_df = pd.DataFrame.from_records(
    [summary_metric, summary_trace, summary_log, summary_error],
    index=['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv']
)

# Keep display compact: ensure lists are not expanded; show at most 20 items for key lists and 50 for cmdb list
summaries_df

```
Out[1]:
```
```python
# Build a concise plain-English summary from the previously computed summaries_df
# (summaries_df exists in the kernel from the prior step)

def first_n_str(lst, n=5):
    if not isinstance(lst, (list, tuple)):
        return str(lst)
    return ", ".join(str(x) for x in lst[:n]) + (", ..." if len(lst) > n else "")

rows_key = 'rows'
min_key = 'min_timestamp'
max_key = 'max_timestamp'
sample_cmdb_key = 'sample_cmdb_ids_(<=50)'
unique_cmdb_key = 'unique_cmdb_count'

lines = []
for idx in ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv']:
    row = summaries_df.loc[idx]
    # determine which "top" column applies for this row
    top_col = next((c for c in ['top_kpi_name_(<=20)', 'top_trace_name_(<=20)', 'top_log_name_(<=20)', 'top_message_(<=20)'] if c in summaries_df.columns and pd.notna(row.get(c))), None)
    top_vals = row[top_col] if top_col is not None and pd.notna(row[top_col]) else []
    lines.append(
        f"{idx}: {int(row[rows_key])} rows; time range {row[min_key]} to {row[max_key]}; "
        f"{int(row[unique_cmdb_key])} unique cmdb_id (sample: {first_n_str(row[sample_cmdb_key], 10)}); "
        f"top keys/messages (sample): {first_n_str(top_vals, 5)}"
    )

summary = "\n".join(lines)

# Display the plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

rows             min_timestamp             max_timestamp                                top_kpi_name_(<=20)                             sample_cmdb_ids_(<=50)  unique_cmdb_count                              top_trace_name_(<=20)               top_log_name_(<=20)                                 top_message_(<=20)
metric.csv      148984 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00  [container.node-5.container_network_receive_pa...  [adservice, adservice-0, adservice-1, adservic...                 59                                                NaN                               NaN                                                NaN
trace.csv        34636 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                                                NaN  [adservice-0, adservice-1, adservice-2, adserv...                 40  [trace.from_frontend-0.duration_mean, trace.fr...                               NaN                                                NaN
log.csv           1758 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                                                NaN  [adservice-0, adservice-1, adservice-2, cartse...                 31                                                NaN  [log.error_count, log.row_count]                                                NaN
error_logs.csv    2060 2022-03-21 18:00:00+08:00 2022-03-21 18:29:59+08:00                                                NaN  [frontend-2, frontend-0, frontend-1, checkouts...                  4                                                NaN                               NaN  [severity: warning, message: failed to retriev...```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing df_metric from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp column as timezone-aware datetimes (UTC->convert to Asia/Shanghai)
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using all available rows (do NOT filter by time)
p95_df = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
p95_df = p95_df.rename(columns={'value': 'p95_threshold'})

# 2) Filter df_metric to the incident window (inclusive)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)
df_window = df_metric[df_metric['ts'].between(start, end)]

# 3) For each pair compute anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p95_threshold
# Merge p95 thresholds into the window rows to identify anomalies
window_with_p95 = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global p95_threshold
window_with_p95['is_anomaly'] = window_with_p95['value'] >= window_with_p95['p95_threshold']

# Aggregates for anomaly rows
anomalies = window_with_p95[window_with_p95['is_anomaly']]
anomaly_stats = anomalies.groupby(['cmdb_id', 'kpi_name'], as_index=True).agg(
    anomaly_count=('is_anomaly', 'sum'),
    earliest_anomaly_timestamp=('ts', 'min')
)

# Max value in the window per pair (even if no anomalies)
max_stats = window_with_p95.groupby(['cmdb_id', 'kpi_name'], as_index=True).agg(
    max_value_in_window=('value', 'max')
)

# Build full per-pair table starting from global p95 (so pairs with no rows in window are included)
metric_anomaly_summary = p95_df.set_index(['cmdb_id', 'kpi_name']).join(max_stats).join(anomaly_stats)

# Ensure columns exist and fill missing values: anomaly_count -> 0, earliest timestamp stays NaT, max_value NaN if absent
metric_anomaly_summary['anomaly_count'] = metric_anomaly_summary['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_timestamp may be NaT for no anomalies (keep as-is)
# p95_threshold exists from p95_df; max_value_in_window may be NaN if no rows in window

# Reset index for display convenience
metric_anomaly_summary = metric_anomaly_summary.reset_index()

# 4) Return compact table: top 20 pairs sorted by anomaly_count desc, tie-breaker earliest_anomaly_timestamp asc
# For sorting, treat NaT as very large so that pairs with no earliest timestamp come after ones with timestamps when tie-breaking
sort_key_time = metric_anomaly_summary['earliest_anomaly_timestamp'].fillna(pd.Timestamp('2100-01-01', tz=tz))
metric_anomaly_summary = metric_anomaly_summary.assign(_sort_time=sort_key_time)

metric_anomaly_summary = metric_anomaly_summary.sort_values(
    by=['anomaly_count', '_sort_time'],
    ascending=[False, True]
).drop(columns=['_sort_time'])

# Keep full table in kernel variable metric_anomaly_summary (already set)
# Prepare compact top-20 view with requested columns
display_cols = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']
metric_anomaly_summary_head = metric_anomaly_summary[display_cols].head(20)

# Display the compact result (per environment rules, return variable)
metric_anomaly_summary_head, metric_anomaly_summary.shape

```
Out[2]:
```
```python
# Summarize the metric anomaly analysis results (plain English) using cached metric_anomaly_summary
tz_str = 'Asia/Shanghai'
import pytz
tz = pytz.timezone(tz_str)

# Use existing metric_anomaly_summary from kernel
total_pairs = metric_anomaly_summary.shape[0]

display_cols = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']
top20 = metric_anomaly_summary[display_cols].head(20).copy()

# quick stats for top20
top20_counts_by_cmdb = top20['cmdb_id'].value_counts()
top_cmdb_summary = ", ".join(f"{cmdb}({cnt})" for cmdb, cnt in top20_counts_by_cmdb.items())

# anomaly_count distribution in top20
anomaly_counts_unique = top20['anomaly_count'].unique().tolist()
anomaly_count_common = top20['anomaly_count'].mode().iloc[0] if not top20['anomaly_count'].empty else None

# earliest anomaly timestamps in top20 (show min)
earliest_ts_min = pd.to_datetime(top20['earliest_anomaly_timestamp']).min()
earliest_ts_min_str = earliest_ts_min.tz_convert(tz).isoformat() if pd.notna(earliest_ts_min) else "N/A"

# sample kpi names from top20 (up to 8)
sample_kpis = ", ".join(top20['kpi_name'].unique()[:8]) + (", ..." if top20['kpi_name'].nunique() > 8 else "")

summary = (
    f"Metric anomaly summary:\n"
    f"- Total (cmdb_id, kpi_name) pairs analyzed: {total_pairs}\n"
    f"- Top 20 pairs (by anomaly_count) are dominated by: {top_cmdb_summary}\n"
    f"- Common anomaly_count in top20: {anomaly_count_common} (unique counts present: {anomaly_counts_unique})\n"
    f"- Earliest anomaly among top20: {earliest_ts_min_str} ({tz_str})\n"
    f"- Sample KPIs in top20: {sample_kpis}\n"
    f"- Observation: many top entries (notably for 'adservice') show anomaly_count = {anomaly_count_common} with earliest anomalies at {earliest_ts_min_str}, "
    f"and their max_value_in_window equals or matches the global P95 threshold. This indicates those KPIs were at or above their global P95 for the entire incident window.\n\n"
    f"Note: Full per-pair results are stored in kernel variable 'metric_anomaly_summary'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_threshold
2   adservice                                        app.grpc.rr             30  2022-03-21 18:00:00+08:00                100.0          100.0
6   adservice                                        app.http.rr             30  2022-03-21 18:00:00+08:00                100.0          100.0
7   adservice                                        app.http.sr             30  2022-03-21 18:00:00+08:00                100.0          100.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount             30  2022-03-21 18:00:00+08:00               5161.0         5161.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             30  2022-03-21 18:00:00+08:00               5187.0         5187.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount             30  2022-03-21 18:00:00+08:00                 26.0           26.0
11  adservice             runtime.java_lang_ClassLoading_Verbose             30  2022-03-21 18:00:00+08:00                  0.0            0.0
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...             30  2022-03-21 18:00:00+08:00                  1.0            1.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...             30  2022-03-21 18:00:00+08:00                  5.0            5.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...             30  2022-03-21 18:00:00+08:00                145.0          145.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00                  1.0            1.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00                  1.0            1.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00                 42.0           42.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00          332354889.0    332354889.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00                  5.0            5.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00           23199744.0     23199744.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00           23134208.0     23134208.0
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00            3932160.0      3932160.0
29  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00            3932160.0      3932160.0
30  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-21 18:00:00+08:00           13238272.0     13238272.0, (5042, 6))```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing df_metric and metric_anomaly_summary if present
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric has timezone-aware datetime column 'ts'
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # Ensure tz-aware and in Asia/Shanghai
    if df_metric['ts'].dt.tz is None:
        df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['ts'] = df_metric['ts'].dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full df_metric (global thresholds BEFORE filtering)
p95_df = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
p95_df = p95_df.rename(columns={'value': 'p95_threshold'})

# 2) Filter df_metric to the incident window (inclusive)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)
df_window = df_metric[df_metric['ts'].between(start, end)].copy()

# 3) Merge p95 thresholds and mark anomalies where value >= p95_threshold
df_window = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
df_window['is_anomaly'] = df_window['value'] >= df_window['p95_threshold']

# Keep only anomaly rows for run detection
anomalies = df_window[df_window['is_anomaly']].copy()

# If there are anomalies, identify consecutive runs per (cmdb_id, kpi_name)
if not anomalies.empty:
    # Sort for reliable diff computation
    anomalies = anomalies.sort_values(['cmdb_id', 'kpi_name', 'ts'])
    # Compute run id: increment when gap != 60 seconds
    def assign_run_id(ts_series):
        diff_sec = ts_series.diff().dt.total_seconds().fillna(9999)
        run_id = (diff_sec != 60).cumsum()
        return run_id

    anomalies['run_id'] = anomalies.groupby(['cmdb_id', 'kpi_name'])['ts'].transform(assign_run_id)

    # Aggregate runs into faults
    metric_faults = anomalies.groupby(['cmdb_id', 'kpi_name', 'run_id'], as_index=False).agg(
        fault_start_timestamp=('ts', 'min'),
        fault_end_timestamp=('ts', 'max'),
        duration_minutes=('ts', 'count'),
        max_value_in_fault=('value', 'max'),
        p95_threshold=('p95_threshold', 'first')
    )

    # 4) Filter out runs shorter than 2 minutes
    metric_faults = metric_faults[metric_faults['duration_minutes'] >= 2].copy()

else:
    # No anomalies -> empty faults table with correct columns
    metric_faults = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'run_id', 'fault_start_timestamp', 'fault_end_timestamp',
        'duration_minutes', 'max_value_in_fault', 'p95_threshold'
    ])

# Cleanup: drop run_id column for final output as it's internal (but keep it in full table if desired)
# The instruction asks to store full faults table in metric_faults, so keep it. For display, select required columns.
display_cols = ['cmdb_id', 'kpi_name', 'fault_start_timestamp', 'fault_end_timestamp',
                'duration_minutes', 'max_value_in_fault', 'p95_threshold']

# 5) Sort and return top 20 faults by duration desc, tie-breaker max_value_in_fault desc
metric_faults = metric_faults.sort_values(by=['duration_minutes', 'max_value_in_fault'], ascending=[False, False]).reset_index(drop=True)

metric_faults_head = metric_faults[display_cols].head(20)

# Return compact result (head) and full faults table shape for context
metric_faults_head, metric_faults.shape

```
Out[3]:
```
```python
# Build a plain-English summary of metric faults using cached metric_faults in kernel
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'metric_faults' not in globals():
    summary = "No metric_faults table found in the kernel."
else:
    mf = metric_faults.copy()
    total_faults = len(mf)
    unique_cmdb = mf['cmdb_id'].nunique()
    top_cmdb = mf['cmdb_id'].value_counts().head(5)
    top_cmdb_str = ", ".join(f"{idx}({cnt})" for idx, cnt in top_cmdb.items())
    max_duration = int(mf['duration_minutes'].max()) if total_faults > 0 else 0
    longest_count = int((mf['duration_minutes'] == max_duration).sum()) if total_faults > 0 else 0

    # Prepare up to 5 sample faults (sorted by duration desc, max_value desc)
    sample = mf.sort_values(by=['duration_minutes', 'max_value_in_fault'], ascending=[False, False]).head(5)
    sample_lines = []
    for _, r in sample.iterrows():
        start = pd.to_datetime(r['fault_start_timestamp'])
        end = pd.to_datetime(r['fault_end_timestamp'])
        # ensure tz and isoformat
        if start.tzinfo is None:
            start = start.tz_localize(tz)
        else:
            start = start.tz_convert(tz)
        if end.tzinfo is None:
            end = end.tz_localize(tz)
        else:
            end = end.tz_convert(tz)
        sample_lines.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | {int(r['duration_minutes'])} min | "
            f"{start.isoformat()} - {end.isoformat()} | max={r['max_value_in_fault']:.6g} | p95={r['p95_threshold']:.6g}"
        )

    sample_text = "\n  - " + "\n  - ".join(sample_lines) if sample_lines else " (no faults)"

    summary = (
        f"Metric faults summary:\n"
        f"- Total faults (>=2 minutes): {total_faults}\n"
        f"- Unique affected cmdb_id count: {unique_cmdb}\n"
        f"- Top affected cmdb_ids (top 5): {top_cmdb_str}\n"
        f"- Longest fault duration: {max_duration} minutes (occurs {longest_count} times)\n"
        f"- Sample top faults (up to 5):{sample_text}\n\n"
        f"Observation: Many top faults span the entire 30-minute incident window (18:00 to 18:29) for various metrics "
        f"(e.g., runtime start times, system.disk.total, container start_time). Full faults table is stored in kernel variable 'metric_faults'."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  duration_minutes  max_value_in_fault  p95_threshold
0                 adservice2                runtime.java_lang_Runtime_StartTime 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647822e+12   1.647822e+12
1                  adservice                runtime.java_lang_Runtime_StartTime 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+12   1.647523e+12
2                     node-4                                  system.disk.total 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        7.616300e+09   7.616300e+09
3                     node-1                                  system.disk.total 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        5.867011e+09   5.867011e+09
4                     node-2                                  system.disk.total 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        5.867011e+09   5.867011e+09
5                     node-3                                  system.disk.total 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        5.867011e+09   5.867011e+09
6                 adservice2  runtime.java_lang_OperatingSystem_CommittedVir... 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        3.566666e+09   3.566666e+09
7                  adservice  runtime.java_lang_OperatingSystem_CommittedVir... 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        3.523752e+09   3.523752e+09
8         currencyservice2-0          container.node-6.container_memory_failcnt 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.659063e+09   1.659063e+09
9             cartservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647857e+09   1.647857e+09
10              adservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647822e+09   1.647822e+09
11             cartservice-0      container.node-5.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647778e+09   1.647778e+09
12             cartservice-1      container.node-5.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647778e+09   1.647778e+09
13             cartservice-2      container.node-5.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647778e+09   1.647778e+09
14  productcatalogservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09
15        currencyservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09
16         paymentservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09
17  recommendationservice2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09
18               frontend2-0      container.node-6.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09
19   productcatalogservice-2      container.node-5.container_start_time_seconds 2022-03-21 18:00:00+08:00 2022-03-21 18:29:00+08:00                30        1.647523e+09   1.647523e+09, (3828, 8))```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace exists and has timezone-aware datetime 'ts'
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if df_trace['ts'].dt.tz is None:
        df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_trace['ts'] = df_trace['ts'].dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire df_trace
p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95)
p95_trace = p95_trace.rename(columns={'value': 'p95_threshold'})

# 2) Filter df_trace to the incident window (inclusive)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)
df_trace_window = df_trace[df_trace['ts'].between(start, end)].copy()

# 3) For each pair within the window compute anomaly_count (value >= p95), earliest_anomaly_timestamp, max_value_in_window, and include p95_threshold
# Merge p95 thresholds into window rows
df_trace_window = df_trace_window.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# Max value in window per pair
max_in_window = df_trace_window.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    max_value_in_window=('value', 'max')
)

# Anomaly rows and their aggregates
anomalies_trace = df_trace_window[df_trace_window['value'] >= df_trace_window['p95_threshold']].copy()

if not anomalies_trace.empty:
    anomaly_stats = anomalies_trace.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_timestamp=('ts', 'min')
    )
else:
    anomaly_stats = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp'])

# Build full per-pair table: start from p95_trace to include all known pairs
trace_anomaly_summary = p95_trace.set_index(['cmdb_id', 'trace_name']).join(
    max_in_window.set_index(['cmdb_id', 'trace_name']), how='left'
).reset_index()

# Join anomaly stats
trace_anomaly_summary = trace_anomaly_summary.merge(anomaly_stats, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing anomaly_count with 0 and keep earliest_anomaly_timestamp as NaT if no anomalies
trace_anomaly_summary['anomaly_count'] = trace_anomaly_summary['anomaly_count'].fillna(0).astype(int)

# 4) Prepare compact top-20 sorted by anomaly_count desc, tie-breaker earliest_anomaly_timestamp asc
# For sorting, treat NaT as very large so those without earliest anomaly come after ones with timestamps
trace_anomaly_summary['_sort_time'] = trace_anomaly_summary['earliest_anomaly_timestamp'].fillna(pd.Timestamp('2100-01-01', tz=tz))

trace_anomaly_summary = trace_anomaly_summary.sort_values(
    by=['anomaly_count', '_sort_time'],
    ascending=[False, True]
).drop(columns=['_sort_time']).reset_index(drop=True)

# Store full table in kernel variable 'trace_anomaly_summary' (already assigned)
# Prepare top-20 compact view
display_cols = ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']
trace_anomaly_top20 = trace_anomaly_summary[display_cols].head(20)

# Also compute total number of unique cmdb_id values present in that top-20 set
unique_cmdb_in_top20 = trace_anomaly_top20['cmdb_id'].nunique()

# Return compact results: top-20 table and the unique cmdb_id count in that top-20
trace_anomaly_top20, unique_cmdb_in_top20

```
Out[4]:
```
```python
# Build a plain-English summary of the trace anomaly analysis using cached trace_anomaly_summary / trace_anomaly_top20
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Prefer to use trace_anomaly_top20 if available, else compute from trace_anomaly_summary
if 'trace_anomaly_top20' in globals():
    top20 = trace_anomaly_top20.copy()
else:
    top20 = trace_anomaly_summary.sort_values(
        by=['anomaly_count', 'earliest_anomaly_timestamp'],
        ascending=[False, True]
    ).head(20)[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']].copy()

total_pairs = trace_anomaly_summary.shape[0] if 'trace_anomaly_summary' in globals() else 0
unique_cmdb_in_top20 = top20['cmdb_id'].nunique()
top_cmdb_counts = top20['cmdb_id'].value_counts().head(5)
top_cmdb_str = ", ".join(f"{idx}({cnt})" for idx, cnt in top_cmdb_counts.items())

# common anomaly_count among top20 and earliest timestamp
common_anomaly_count = int(top20['anomaly_count'].mode().iloc[0]) if not top20['anomaly_count'].empty else 0
earliest_ts = pd.to_datetime(top20['earliest_anomaly_timestamp']).min()
earliest_ts_str = earliest_ts.tz_convert(tz).isoformat() if pd.notna(earliest_ts) else "N/A"

# Check for p95 values that are zero in top20
zero_p95_count = int((top20['p95_threshold'] == 0).sum())

summary = (
    f"Trace anomaly summary:\n"
    f"- Total (cmdb_id, trace_name) pairs analyzed: {total_pairs}\n"
    f"- Top 20 pairs (by anomaly_count) include {unique_cmdb_in_top20} unique cmdb_id(s); top examples: {top_cmdb_str}\n"
    f"- In the top 20, the common anomaly_count = {common_anomaly_count}; earliest anomaly seen = {earliest_ts_str} (Asia/Shanghai)\n"
    f"- Notably, {zero_p95_count} of the top-20 pairs have a P95 threshold of 0 and many max values are also 0. Many top entries (e.g., trace.from_frontend-*.error_rate) show anomaly_count = {common_anomaly_count} across the whole incident window.\n"
    f"- Observation: the results indicate many trace metrics were flagged for the entire incident window because their global P95 is 0 (so value >= P95 is trivially true). This suggests thresholding or metric semantics should be reviewed before treating these as meaningful anomalies.\n\n"
    f"Full per-pair results are stored in kernel variable 'trace_anomaly_summary'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                  trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_threshold
0     adservice-0            trace.from_frontend-0.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
1     adservice-0            trace.from_frontend-1.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
2     adservice-0            trace.from_frontend-2.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
3     adservice-1            trace.from_frontend-0.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
4     adservice-1            trace.from_frontend-1.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
5     adservice-1            trace.from_frontend-2.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
6     adservice-2            trace.from_frontend-0.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
7     adservice-2            trace.from_frontend-1.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
8     adservice-2            trace.from_frontend-2.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
9    adservice2-0           trace.from_frontend2-0.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
10  cartservice-0  trace.from_checkoutservice-2.duration_mean             30  2022-03-21 18:00:00+08:00              0.00000            0.0
11  cartservice-0   trace.from_checkoutservice-2.duration_p95             30  2022-03-21 18:00:00+08:00              0.00000            0.0
12  cartservice-0     trace.from_checkoutservice-2.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
13  cartservice-0          trace.from_frontend-0.duration_p95             30  2022-03-21 18:00:00+08:00              0.00000            0.0
14  cartservice-0            trace.from_frontend-0.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
15  cartservice-0          trace.from_frontend-1.duration_p95             30  2022-03-21 18:00:00+08:00              0.00010            0.0
16  cartservice-0            trace.from_frontend-1.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
17  cartservice-0          trace.from_frontend-2.duration_p95             30  2022-03-21 18:00:00+08:00              0.00095            0.0
18  cartservice-0            trace.from_frontend-2.error_rate             30  2022-03-21 18:00:00+08:00              0.00000            0.0
19  cartservice-0                     trace.self.duration_p95             30  2022-03-21 18:00:00+08:00              0.00000            0.0, 5)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_error exists and has timezone-aware datetime 'ts'
if 'ts' not in df_error.columns:
    df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_error['ts'].dt.tz is None:
        df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_error['ts'] = df_error['ts'].dt.tz_convert(tz)

# 1) Total message count and number of unique cmdb_id
total_messages = len(df_error)
unique_cmdb_count = int(df_error['cmdb_id'].nunique())

# 2) Per-cmdb_id aggregates: message_count, earliest, latest; top 20 by message_count desc
per_cmdb = (
    df_error.groupby('cmdb_id', as_index=False)
    .agg(
        message_count=('message', 'count'),
        earliest_message_timestamp=('ts', 'min'),
        latest_message_timestamp=('ts', 'max')
    )
    .sort_values(by='message_count', ascending=False)
)
top20_cmdb = per_cmdb.head(20)

# 3) Per-minute aggregation within incident window (inclusive)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)

error_window = df_error[df_error['ts'].between(start, end)].copy()
# Floor to minute to be explicit
error_window['minute'] = error_window['ts'].dt.floor('T')

error_logs_timeseries = (
    error_window.groupby('minute', as_index=True)
    .size()
    .rename('count')
    .reset_index()
    .sort_values('minute')
)

# Keep only minutes with count > 0 (groupby already yields only present minutes)
# Store full per-minute series in kernel variable error_logs_timeseries (done above)
# For display, limit to top 50 rows
error_logs_timeseries_head = error_logs_timeseries[error_logs_timeseries['count'] > 0].head(50)

# 4) Top 20 unique message text samples overall (truncate to 200 chars) with counts
message_counts = (
    df_error.groupby('message', as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
)

# Prepare display samples truncated to 200 chars
message_counts['message_sample'] = message_counts['message'].astype(str).str.slice(0, 200)
top20_messages = message_counts[['message_sample', 'count']].head(20)

# Compact outputs (per instructions, return variables rather than print)
total_messages, unique_cmdb_count, top20_cmdb[['cmdb_id','message_count','earliest_message_timestamp','latest_message_timestamp']], error_logs_timeseries_head, top20_messages

```
Out[5]:
```
```python
# Build a concise plain-English summary of the error_logs analysis using cached data in the kernel
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cached df_error and error_logs_timeseries if present; otherwise compute minimal values
# (these variables were computed earlier in the session)
try:
    df = df_error
except NameError:
    df = pd.read_csv('error_logs.csv')
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Total messages and unique cmdb_id
total_messages = len(df)
unique_cmdb = int(df['cmdb_id'].nunique())

# Per-cmdb summary (top 20 by message count)
per_cmdb = (
    df.groupby('cmdb_id', as_index=False)
      .agg(message_count=('message', 'count'),
           earliest_message_timestamp=('timestamp', lambda s: pd.to_datetime(s.iloc[0], unit='s', utc=True).tz_convert(tz) if len(s)>0 else pd.NaT),
           latest_message_timestamp=('timestamp', lambda s: pd.to_datetime(s.iloc[-1], unit='s', utc=True).tz_convert(tz) if len(s)>0 else pd.NaT))
)
# More accurate earliest/latest using ts if available
if 'ts' in df.columns:
    per_cmdb = (
        df.groupby('cmdb_id', as_index=False)
          .agg(message_count=('message', 'count'),
               earliest_message_timestamp=('ts', 'min'),
               latest_message_timestamp=('ts', 'max'))
    )

top20_cmdb = per_cmdb.sort_values('message_count', ascending=False).head(20)

# Per-minute series within incident window (use cached error_logs_timeseries if available)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)

if 'error_logs_timeseries' in globals():
    ts_df = error_logs_timeseries.copy()
else:
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window = df[df['ts'].between(start, end)].copy()
    window['minute'] = window['ts'].dt.floor('T')
    ts_df = window.groupby('minute', as_index=True).size().rename('count').reset_index().sort_values('minute')
    # store in kernel variable as requested
    error_logs_timeseries = ts_df

# Filter to minutes with count>0 and limit to 50 rows
ts_nonzero = ts_df[ts_df['count'] > 0].sort_values('minute').head(50)

# Top message text samples (truncate to 200 chars) with counts
msg_counts = df.groupby('message', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False)
msg_counts['sample'] = msg_counts['message'].astype(str).str.slice(0, 200)
top20_messages = msg_counts[['sample','count']].head(20)

# Find peak minute if present
peak_row = ts_nonzero.loc[ts_nonzero['count'].idxmax()] if not ts_nonzero.empty else None
peak_minute_str = peak_row['minute'].isoformat() if peak_row is not None else "N/A"
peak_count = int(peak_row['count']) if peak_row is not None else 0

# Prepare the plain-English summary
summary = (
    f"Error logs compact summary:\n"
    f"- Total log messages: {total_messages}\n"
    f"- Unique cmdb_id count: {unique_cmdb}\n\n"
    f"- Top cmdb_id by message_count (top shown):\n"
)

# Add top 3 cmdb examples (if present)
top_examples = []
for _, r in top20_cmdb.head(5).iterrows():
    start_ts = r['earliest_message_timestamp']
    end_ts = r['latest_message_timestamp']
    # format timestamps consistently
    start_str = start_ts.isoformat() if pd.notna(start_ts) else "N/A"
    end_str = end_ts.isoformat() if pd.notna(end_ts) else "N/A"
    top_examples.append(f"  - {r['cmdb_id']}: {int(r['message_count'])} messages, earliest={start_str}, latest={end_str}")

if not top_examples:
    summary += "  (no cmdb data)\n"
else:
    summary += "\n".join(top_examples) + "\n"

# Per-minute series summary
summary += (
    f"\n- Per-minute counts within 2022-03-21 18:00:00+08:00 through 18:30:00+08:00 (minutes with >0 counts shown, up to 50 rows):\n"
    f"  - Total minutes with logs in window: {len(ts_nonzero)}\n"
    f"  - Peak minute: {peak_minute_str} with {peak_count} messages\n"
    f"  - Typical per-minute counts in the window observed around ~58–70, with a final spike at 18:29 having {peak_count} messages.\n"
)

# Top message samples
summary += "\n- Top message samples (truncated to 200 chars) with counts:\n"
for _, row in top20_messages.iterrows():
    msg = row['sample']
    cnt = int(row['count'])
    summary += f"  - ({cnt}) {msg}\n"

summary += (
    "\nObservation:\n"
    "- The majority of messages (~1936 of 2060) are the warning 'failed to retrieve...' coming from frontend nodes (frontend-0/1/2).\n"
    "- Frontend components account for the bulk of logs during the incident window; investigate frontend interactions and upstream dependencies.\n"
    "- The per-minute time series shows sustained activity across the whole 30-minute window with a notable peak at 18:29.\n\n"
    "Stored artifacts:\n"
    "- Full per-minute series stored in kernel variable 'error_logs_timeseries'.\n"
    "- Full error logs dataframe is available as 'df_error'."
)

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(2060, 4,              cmdb_id  message_count earliest_message_timestamp  latest_message_timestamp
1         frontend-0            965  2022-03-21 18:00:01+08:00 2022-03-21 18:29:59+08:00
2         frontend-1            556  2022-03-21 18:00:02+08:00 2022-03-21 18:29:59+08:00
3         frontend-2            537  2022-03-21 18:00:00+08:00 2022-03-21 18:29:59+08:00
0  checkoutservice-2              2  2022-03-21 18:11:41+08:00 2022-03-21 18:12:49+08:00,                       minute  count
0  2022-03-21 18:00:00+08:00     59
1  2022-03-21 18:01:00+08:00     65
2  2022-03-21 18:02:00+08:00     60
3  2022-03-21 18:03:00+08:00     70
4  2022-03-21 18:04:00+08:00     58
..                       ...    ...
25 2022-03-21 18:25:00+08:00     65
26 2022-03-21 18:26:00+08:00     61
27 2022-03-21 18:27:00+08:00     64
28 2022-03-21 18:28:00+08:00     58
29 2022-03-21 18:29:00+08:00    163

[30 rows x 2 columns],                                        message_sample  count
61  severity: warning, message: failed to retrieve...   1936
60            severity: error, message: request error     62
62  severity: warning, message: failed to send ord...      2
3   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
0   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
1   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
2   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
7   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
8   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
9   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
10  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
11  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
4   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
5   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
6   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
15  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
16  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
17  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
18  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
19  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_log exists in kernel and has timezone-aware 'ts' column
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if df_log['ts'].dt.tz is None:
        df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_log['ts'] = df_log['ts'].dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, log_name) using the full series (do NOT filter by time)
p95_log = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95)
p95_log = p95_log.rename(columns={'value': 'p95_threshold'})

# 2) Filter df_log to the incident window (inclusive)
start = pd.Timestamp('2022-03-21 18:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 18:30:00', tz=tz)
df_log_window = df_log[df_log['ts'].between(start, end)].copy()

# 3) For each pair within the window compute anomaly_count (value >= P95), earliest_anomaly_timestamp, max_value_in_window, p95_threshold
# Merge p95 thresholds into window rows
df_log_window = df_log_window.merge(p95_log, on=['cmdb_id', 'log_name'], how='left')

# Max value in window per pair
max_in_window = df_log_window.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    max_value_in_window=('value', 'max')
)

# Anomaly rows and their aggregates
anomalies_log = df_log_window[df_log_window['value'] >= df_log_window['p95_threshold']].copy()

if not anomalies_log.empty:
    anomaly_stats = anomalies_log.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_timestamp=('ts', 'min')
    )
else:
    anomaly_stats = pd.DataFrame(columns=['cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_timestamp'])

# Build full per-pair table starting from p95_log so we include all known pairs (even if not present in window)
log_anomaly_summary = p95_log.set_index(['cmdb_id', 'log_name']).join(
    max_in_window.set_index(['cmdb_id', 'log_name']), how='left'
).reset_index()

# Join anomaly stats
log_anomaly_summary = log_anomaly_summary.merge(anomaly_stats, on=['cmdb_id', 'log_name'], how='left')

# Fill missing anomaly_count with 0 and keep earliest_anomaly_timestamp as NaT if no anomalies
log_anomaly_summary['anomaly_count'] = log_anomaly_summary['anomaly_count'].fillna(0).astype(int)

# 4) Prepare compact top-20 sorted by anomaly_count desc, tie-breaker earliest_anomaly_timestamp asc
log_anomaly_summary['_sort_time'] = log_anomaly_summary['earliest_anomaly_timestamp'].fillna(pd.Timestamp('2100-01-01', tz=tz))
log_anomaly_summary = log_anomaly_summary.sort_values(by=['anomaly_count', '_sort_time'], ascending=[False, True]).drop(columns=['_sort_time']).reset_index(drop=True)

# Store full per-pair results in kernel variable (already assigned)
# Prepare top-20 compact view with requested columns
display_cols = ['cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']
log_anomaly_top20 = log_anomaly_summary[display_cols].head(20)

# Return compact table and shape of full summary
log_anomaly_top20, log_anomaly_summary.shape

```
Out[6]:
```
```python
# Summarize the log anomaly analysis results (plain English) using cached variables in the kernel
# The summary will be displayed as a single concise string.

# Use existing variables computed earlier: log_anomaly_summary, log_anomaly_top20
summary = (
    "Log anomaly summary:\n"
    "- Total (cmdb_id, log_name) pairs analyzed (full table): 62\n"
    "- Top 20 pairs (by anomaly_count) are shown in 'log_anomaly_top20'.\n"
    "- Observation from the top-20: every entry is 'log.error_count' for various services with:\n"
    "  * anomaly_count = 30 (i.e., every minute in the 30-minute incident window was flagged),\n"
    "  * earliest_anomaly_timestamp = 2022-03-21 18:00:00+08:00,\n"
    "  * max_value_in_window = 0.0, p95_threshold = 0.0.\n"
    "- Interpretation: the global P95 for these log.error_count series is 0, so value >= P95 is trivially true and marks all rows as anomalies. "
    "This indicates the thresholding rule (using raw P95=0) is producing many false-positive anomalies for log.error_count. "
    "Recommend reviewing metric semantics and using a different anomaly rule or minimum threshold before treating these as real incidents.\n"
    "- Actionable notes: full per-pair results are stored in kernel variable 'log_anomaly_summary'. Use that for further inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_threshold
0               adservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
1               adservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
2               adservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
3             cartservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
4             cartservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
5             cartservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
6         checkoutservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
7         checkoutservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
8         currencyservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
9         currencyservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
10        currencyservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
11  productcatalogservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
12  productcatalogservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
13  productcatalogservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
14  recommendationservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
15  recommendationservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
16  recommendationservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
17        shippingservice-0  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
18        shippingservice-1  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0
19        shippingservice-2  log.error_count             30  2022-03-21 18:00:00+08:00                  0.0            0.0, (62, 6))```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_anomaly_summary exists
if 'trace_anomaly_summary' not in globals():
    # If missing, create an empty DataFrame to avoid runtime error and produce empty outputs
    trace_anomaly_summary = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_threshold'])

# Ensure earliest_anomaly_timestamp is timezone-aware (convert if present and naive)
if 'earliest_anomaly_timestamp' in trace_anomaly_summary.columns and not trace_anomaly_summary.empty:
    # convert to datetime if not already
    trace_anomaly_summary['earliest_anomaly_timestamp'] = pd.to_datetime(trace_anomaly_summary['earliest_anomaly_timestamp'])
    # localize/convert: if timezone-naive, localize as Asia/Shanghai; if tz-aware, convert to Asia/Shanghai
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    trace_anomaly_summary['earliest_anomaly_timestamp'] = trace_anomaly_summary['earliest_anomaly_timestamp'].apply(_ensure_tz)

# 1) Filter rows where p95_threshold > 0 and anomaly_count > 0
trace_anomaly_informative = trace_anomaly_summary[
    (trace_anomaly_summary['p95_threshold'].astype(float) > 0) &
    (trace_anomaly_summary['anomaly_count'].astype(int) > 0)
].copy()

# 2) Sorting: anomaly_count desc, earliest_anomaly_timestamp asc (treat NaT as far future)
if 'earliest_anomaly_timestamp' in trace_anomaly_informative.columns:
    sort_time = trace_anomaly_informative['earliest_anomaly_timestamp'].fillna(pd.Timestamp('2100-01-01', tz=tz))
else:
    sort_time = pd.Series(pd.NaT, index=trace_anomaly_informative.index).fillna(pd.Timestamp('2100-01-01', tz=tz))

trace_anomaly_informative = trace_anomaly_informative.assign(_sort_time=sort_time)
trace_anomaly_informative = trace_anomaly_informative.sort_values(by=['anomaly_count', '_sort_time'], ascending=[False, True]).drop(columns=['_sort_time'])

# 3) Prepare compact top-20 view with requested columns
display_cols = ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p95_threshold']
trace_anomaly_informative_head = trace_anomaly_informative[display_cols].head(20)

# 4) Number of unique cmdb_id in the filtered set
unique_cmdb_in_informative = int(trace_anomaly_informative['cmdb_id'].nunique())

# Return compact results (variable names are used so IPython displays them)
trace_anomaly_informative_head, unique_cmdb_in_informative

```
Out[7]:
```
```python
# Plain-English summary of trace_anomaly_informative (already cached in kernel)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use the cached DataFrame
df = trace_anomaly_informative if 'trace_anomaly_informative' in globals() else pd.DataFrame()

total_rows = int(df.shape[0])
unique_cmdb = int(df['cmdb_id'].nunique()) if not df.empty else 0

# Top affected cmdb_id (top 5)
top_cmdb_series = df['cmdb_id'].value_counts().head(5)
top_cmdb_list = [f"{idx}({cnt})" for idx, cnt in top_cmdb_series.items()]

# Sample top rows (up to 10) for concise examples
sample_rows = []
if not df.empty:
    sample = df.sort_values(by=['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True]).head(10)
    for _, r in sample.iterrows():
        ts = r['earliest_anomaly_timestamp']
        ts_str = ts.isoformat() if pd.notna(ts) else "N/A"
        sample_rows.append(
            f"{r['cmdb_id']} | {r['trace_name']} | anomalies={int(r['anomaly_count'])} | earliest={ts_str} | max={r['max_value_in_window']} | p95={r['p95_threshold']}"
        )

summary_lines = [
    "Trace anomalies (informative subset where P95>0 and anomaly_count>0):",
    f"- Total matching rows: {total_rows}",
    f"- Unique cmdb_id in this set: {unique_cmdb}",
    f"- Top affected cmdb_id (top 5): {', '.join(top_cmdb_list) if top_cmdb_list else 'N/A'}",
    "- Sample top anomaly entries (up to 10):"
]
summary_lines += [f"  - {s}" for s in sample_rows] if sample_rows else ["  (no rows)"]

summary_lines += [
    "- Observations:",
    "  * Several trace metrics show sustained anomalies across the incident window (many have anomaly_count=30 or near-30, earliest at 2022-03-21 18:00:00+08:00).",
    "  * Notable metric groups: duration_p95 and row_count series (examples: cartservice2-0 duration_p95; many checkoutservice/* and emailservice/* row_count traces).",
    "  * The P95 thresholds retained here are >0, so these detections are potentially meaningful; investigate the listed cmdb_id/trace_name pairs (especially those with high anomaly_count and multi-minute duration).",
    "",
    "Full filtered DataFrame is stored in kernel variable 'trace_anomaly_informative'."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                              trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_threshold
36      cartservice2-0     trace.from_frontend2-0.duration_p95             30  2022-03-21 18:00:00+08:00                0.001          0.001
129        frontend2-0    trace.to_cartservice2-0.duration_p95             30  2022-03-21 18:00:00+08:00                0.001          0.001
283  checkoutservice-1         trace.from_frontend-0.row_count             26  2022-03-21 18:00:00+08:00                3.000          2.000
295         frontend-0    trace.to_checkoutservice-1.row_count             26  2022-03-21 18:00:00+08:00                3.000          2.000
361  checkoutservice-1         trace.from_frontend-2.row_count             12  2022-03-21 18:00:00+08:00                2.000          2.000
362  checkoutservice-1     trace.to_paymentservice-2.row_count             12  2022-03-21 18:00:00+08:00                2.000          2.000
363         frontend-2    trace.to_checkoutservice-1.row_count             12  2022-03-21 18:00:00+08:00                2.000          2.000
364   paymentservice-2  trace.from_checkoutservice-1.row_count             12  2022-03-21 18:00:00+08:00                2.000          2.000
365  checkoutservice-0       trace.to_emailservice-2.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
366  checkoutservice-0     trace.to_paymentservice-2.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
367  checkoutservice-1         trace.from_frontend-1.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
368  checkoutservice-1       trace.to_emailservice-0.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
369  checkoutservice-1       trace.to_emailservice-2.row_count             11  2022-03-21 18:00:00+08:00                3.000          2.000
370  checkoutservice-2       trace.to_emailservice-0.row_count             11  2022-03-21 18:00:00+08:00                3.000          2.000
371  checkoutservice-2       trace.to_emailservice-2.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
372  checkoutservice-2     trace.to_paymentservice-1.row_count             11  2022-03-21 18:00:00+08:00                3.000          2.000
373     emailservice-0  trace.from_checkoutservice-1.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
374     emailservice-0  trace.from_checkoutservice-2.row_count             11  2022-03-21 18:00:00+08:00                3.000          2.000
375     emailservice-2  trace.from_checkoutservice-0.row_count             11  2022-03-21 18:00:00+08:00                2.000          2.000
376     emailservice-2  trace.from_checkoutservice-1.row_count             11  2022-03-21 18:00:00+08:00                3.000          2.000, 40)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_faults exists in the kernel
if 'metric_faults' not in globals():
    metric_faults = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'fault_start_timestamp', 'fault_end_timestamp',
        'duration_minutes', 'max_value_in_fault', 'p95_threshold'
    ])

# Ensure fault_start_timestamp is timezone-aware Asia/Shanghai
if 'fault_start_timestamp' in metric_faults.columns and not metric_faults.empty:
    metric_faults['fault_start_timestamp'] = pd.to_datetime(metric_faults['fault_start_timestamp'])
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    metric_faults['fault_start_timestamp'] = metric_faults['fault_start_timestamp'].apply(_ensure_tz)

# Ensure numeric columns exist
metric_faults['max_value_in_fault'] = pd.to_numeric(metric_faults['max_value_in_fault'], errors='coerce').fillna(0.0)
metric_faults['p95_threshold'] = pd.to_numeric(metric_faults['p95_threshold'], errors='coerce').fillna(0.0)
metric_faults['duration_minutes'] = pd.to_numeric(metric_faults['duration_minutes'], errors='coerce').fillna(0).astype(int)

# Compute severity per rules:
# if p95_threshold > 0: severity = (max_value_in_fault - p95_threshold) / p95_threshold
# elif p95_threshold == 0 and max_value_in_fault > 0: severity = 1e6
# else severity = 0
p95 = metric_faults['p95_threshold']
maxv = metric_faults['max_value_in_fault']

severity = np.where(
    p95 > 0,
    (maxv - p95) / p95,
    np.where((p95 == 0) & (maxv > 0), 1e6, 0.0)
)

metric_faults_severity = metric_faults.copy()
metric_faults_severity['severity'] = severity.astype(float)

# Sort by severity desc, tie-breaker duration_minutes desc
metric_faults_severity = metric_faults_severity.sort_values(by=['severity', 'duration_minutes'], ascending=[False, False]).reset_index(drop=True)

# Prepare compact top-20 view with requested columns
display_cols = ['cmdb_id', 'kpi_name', 'fault_start_timestamp', 'duration_minutes', 'max_value_in_fault', 'p95_threshold', 'severity']
metric_faults_severity_head = metric_faults_severity[display_cols].head(20)

# Count node-level vs others in the top-20
top20 = metric_faults_severity_head
node_count = int(top20['cmdb_id'].str.startswith('node-').sum())
other_count = int(len(top20) - node_count)

# Expose variables to kernel: metric_faults_severity (full augmented DF), and compact outputs
metric_faults_severity_head, node_count, other_count

```
Out[8]:
```
```python
# Build a concise plain-English summary of the severity-ranked faults using cached metric_faults_severity in kernel
import pandas as pd

# Use existing augmented faults table
mf = metric_faults_severity if 'metric_faults_severity' in globals() else None

if mf is None or mf.empty:
    summary = "No metric_faults_severity data available in the kernel."
else:
    top20 = mf.head(20).copy()
    # Count node-level vs others in the top-20
    node_count = int(top20['cmdb_id'].str.startswith('node-').sum())
    other_count = int(len(top20) - node_count)
    # How many have p95_threshold == 0 among top20
    zero_p95 = int((top20['p95_threshold'] == 0).sum())
    # Typical duration among top20 (most common)
    common_duration = int(top20['duration_minutes'].mode().iloc[0]) if not top20['duration_minutes'].empty else 0
    # Top 5 examples
    examples = []
    for _, r in top20.head(5).iterrows():
        examples.append(f"{r['cmdb_id']} | {r['kpi_name']} | duration={int(r['duration_minutes'])}m | max={r['max_value_in_fault']:.6g} | p95={r['p95_threshold']:.6g} | severity={r['severity']:.6g}")
    examples_text = "\n  - ".join(examples)

    summary = (
        "Metric faults severity summary:\n"
        f"- Top-20 faults are dominated by non-node components: node-level={node_count}, others={other_count}.\n"
        f"- Many top entries have p95_threshold == 0 (count={zero_p95}), so severity was set to a large constant (1e6) when max_value_in_fault > 0.\n"
        f"- Typical fault duration among top-20: {common_duration} minutes (many are 30 minutes covering the incident window).\n"
        "- Top example faults (up to 5):\n"
        f"  - {examples_text}\n\n"
        "Interpretation and recommendation:\n"
        "- A large portion of high-severity faults come from mesh/HTTP metrics (istio request/response counters) reported on frontend and adservice components.\n"
        "- Because many p95 thresholds are zero, the severity rule flagged absolute breaches rather than relative deviations. Review metric semantics and thresholds (do not treat p95==0 as a meaningful baseline).\n"
        "- Prioritize investigating frontend/adservice mesh traffic patterns and why these istio metrics are high during the incident window; but first adjust anomaly rules to avoid false-positive severity from p95==0.\n\n"
        "Notes: full augmented faults table is stored in kernel variable 'metric_faults_severity'."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name     fault_start_timestamp  duration_minutes  max_value_in_fault  p95_threshold   severity
0    frontend2-0  mesh.source.frontend2.adservice2.istio_request... 2022-03-21 18:00:00+08:00                30             23250.0            0.0  1000000.0
1     frontend-0  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30             22025.0            0.0  1000000.0
2     frontend-2  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30             16275.0            0.0  1000000.0
3    frontend2-0  mesh.source.frontend2.adservice2.istio_request... 2022-03-21 18:00:00+08:00                30             15500.0            0.0  1000000.0
4     frontend-0  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30             15300.0            0.0  1000000.0
5     frontend-1  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30             10075.0            0.0  1000000.0
6     frontend-1  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30              9300.0            0.0  1000000.0
7   adservice2-0  mesh.destination.frontend2.adservice2.istio_re... 2022-03-21 18:00:00+08:00                30              7312.5            0.0  1000000.0
8     frontend-2  mesh.source.frontend.adservice.istio_request_b... 2022-03-21 18:00:00+08:00                30              6975.0            0.0  1000000.0
9    adservice-2  mesh.destination.frontend.adservice.istio_resp... 2022-03-21 18:00:00+08:00                30              4600.0            0.0  1000000.0
10    frontend-1  mesh.source.frontend.checkoutservice.istio_req... 2022-03-21 18:00:00+08:00                30              3300.0            0.0  1000000.0
11   adservice-0  mesh.destination.frontend.adservice.istio_resp... 2022-03-21 18:00:00+08:00                30              2875.0            0.0  1000000.0
12   adservice-2  mesh.destination.frontend.adservice.istio_requ... 2022-03-21 18:00:00+08:00                30              2540.0            0.0  1000000.0
13   frontend2-0  mesh.source.frontend2.adservice2.istio_respons... 2022-03-21 18:00:00+08:00                30              2025.0            0.0  1000000.0
14    frontend-0  mesh.source.frontend.adservice.istio_response_... 2022-03-21 18:00:00+08:00                30              1957.5            0.0  1000000.0
15   adservice-0  mesh.source.adservice.jaeger-collector.istio_r... 2022-03-21 18:00:00+08:00                30              1750.0            0.0  1000000.0
16   adservice-1  mesh.destination.frontend.adservice.istio_resp... 2022-03-21 18:00:00+08:00                30              1725.0            0.0  1000000.0
17  adservice2-0  mesh.destination.frontend2.adservice2.istio_re... 2022-03-21 18:00:00+08:00                30              1725.0            0.0  1000000.0
18   adservice-0  mesh.destination.frontend.adservice.istio_requ... 2022-03-21 18:00:00+08:00                30              1582.5            0.0  1000000.0
19    frontend-2  mesh.source.frontend.adservice.istio_response_... 2022-03-21 18:00:00+08:00                30              1417.5            0.0  1000000.0, 0, 20)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing metric_faults_severity from kernel
if 'metric_faults_severity' not in globals():
    metric_faults_severity = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'duration_minutes','max_value_in_fault','p95_threshold','severity'
    ])

# Ensure numeric types and timezone for timestamp
metric_faults_severity['p95_threshold'] = pd.to_numeric(metric_faults_severity['p95_threshold'], errors='coerce').fillna(0.0)
metric_faults_severity['severity'] = pd.to_numeric(metric_faults_severity['severity'], errors='coerce').fillna(0.0)
metric_faults_severity['max_value_in_fault'] = pd.to_numeric(metric_faults_severity['max_value_in_fault'], errors='coerce').fillna(0.0)
metric_faults_severity['duration_minutes'] = pd.to_numeric(metric_faults_severity['duration_minutes'], errors='coerce').fillna(0).astype(int)

if 'fault_start_timestamp' in metric_faults_severity.columns and not metric_faults_severity.empty:
    metric_faults_severity['fault_start_timestamp'] = pd.to_datetime(metric_faults_severity['fault_start_timestamp'])
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    metric_faults_severity['fault_start_timestamp'] = metric_faults_severity['fault_start_timestamp'].apply(_ensure_tz)

# Filter: p95_threshold > 0 and severity > 0 (exclude artificial large severity for p95==0)
metric_faults_real_severity = metric_faults_severity[
    (metric_faults_severity['p95_threshold'] > 0) &
    (metric_faults_severity['severity'] > 0)
].copy()

# Sort by severity desc, tie-breaker duration_minutes desc
metric_faults_real_severity = metric_faults_real_severity.sort_values(
    by=['severity', 'duration_minutes'],
    ascending=[False, False]
).reset_index(drop=True)

# Prepare compact top-20 view with requested columns
display_cols = ['cmdb_id', 'kpi_name', 'fault_start_timestamp', 'duration_minutes', 'max_value_in_fault', 'p95_threshold', 'severity']
metric_faults_real_severity_head = metric_faults_real_severity[display_cols].head(20)

# Summary counts
real_count = int(metric_faults_real_severity.shape[0])

# Counts of node-level vs others among the top-20 displayed
top20 = metric_faults_real_severity_head
node_count_top20 = int(top20['cmdb_id'].str.startswith('node-').sum())
other_count_top20 = int(len(top20) - node_count_top20)

# Return compact outputs (variables displayed by IPython)
metric_faults_real_severity_head, real_count, node_count_top20, other_count_top20

```
Out[9]:
```
```python
# Build a concise plain-English summary of the filtered real-severity faults
import pandas as pd

# Use cached metric_faults_real_severity if available
mf = metric_faults_real_severity if 'metric_faults_real_severity' in globals() else pd.DataFrame()

if mf.empty:
    summary = "No faults with p95_threshold>0 and severity>0 found in the kernel variable 'metric_faults_real_severity'."
else:
    total_rows = int(mf.shape[0])
    # Top-20 by severity (already sorted in the variable used earlier); take up to 20
    top20 = mf.sort_values(by=['severity', 'duration_minutes'], ascending=[False, False]).head(20).copy()
    # Count node-level vs others among these top-20
    node_count_top20 = int(top20['cmdb_id'].str.startswith('node-').sum())
    other_count_top20 = int(len(top20) - node_count_top20)
    # Prepare top example lines (top 6)
    examples = []
    for _, r in top20.head(6).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | duration={int(r['duration_minutes'])}m | "
            f"max={float(r['max_value_in_fault']):.6g} | p95={float(r['p95_threshold']):.6g} | severity={float(r['severity']):.6g}"
        )
    examples_text = "\n  - ".join(examples)

    summary_lines = [
        "Filtered metric faults (p95_threshold > 0 and severity > 0) — concise summary:",
        f"- Total faults matching filter: {total_rows}",
        f"- Top-20 breakdown: node-level = {node_count_top20}, pod/service-level = {other_count_top20}",
        f"- Most faults are short runs (many with duration = 2 minutes).",
        "- Top example faults (up to 6):",
        f"  - {examples_text}",
        "",
        "Key observations and suggested next steps:",
        "- Highest severities affect checkoutservice mesh metrics (e.g., jaeger/istio counters) and some mesh destination metrics for frontend-checkout paths — investigate checkoutservice-related traffic and tracing/collector behavior around 18:14.",
        "- Node-level signals appear too (e.g., node-5 CPU iowait and node-6 TCP retransmission) — check node-5 I/O and node-6 network for packet loss/retransmissions around ~18:02 and 18:26.",
        "- Some container I/O and memory-failure related metrics on paymentservice2-0 are flagged — inspect container disk read activity and memory failure counters on that node/pod.",
        "- Because many durations are 2 minutes, these may be short but consistent breaches; determine whether they correlate with specific requests/traffic spikes or transient infra issues.",
        "",
        "Stored artifact: the full filtered table is saved in kernel variable 'metric_faults_real_severity' for detailed inspection."
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name     fault_start_timestamp  duration_minutes  max_value_in_fault  p95_threshold      severity
0   checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i... 2022-03-21 18:14:00+08:00                 2          126.047100       0.002467  51081.917934
1   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2          475.850000       6.200000     75.750000
2              node-5                                  system.cpu.iowait 2022-03-21 18:26:00+08:00                 2            4.210000       0.060000     69.166667
3   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2        16092.500000     314.000000     50.250000
4   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2        28825.000000     580.000000     48.698276
5     checkoutservice                                       app.grpc.mrt 2022-03-21 18:14:00+08:00                 2           21.206364       1.073600     18.752575
6      emailservice-1  container.node-5.container_network_receive_MB.... 2022-03-21 18:16:00+08:00                 2            0.555434       0.028484     18.500170
7   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i... 2022-03-21 18:14:00+08:00                 2          172.611050      22.203750      6.773959
8              node-6                        system.net.tcp.retrans_segs 2022-03-21 18:02:00+08:00                 2            0.670000       0.158000      3.240506
9   paymentservice2-0    container.node-6.container_fs_reads_MB./dev/vda 2022-03-21 18:14:00+08:00                 2            0.111979       0.030794      2.636364
10  paymentservice2-0       container.node-6.container_fs_reads./dev/vda 2022-03-21 18:14:00+08:00                 2            5.333333       1.466667      2.636364
11  paymentservice2-0  container.node-6.container_memory_failures.con... 2022-03-21 18:14:00+08:00                 2            0.666667       0.183333      2.636364
12  paymentservice2-0  container.node-6.container_memory_failures.hie... 2022-03-21 18:14:00+08:00                 2            0.666667       0.183333      2.636364
13             node-1                                    system.cpu.user 2022-03-21 18:02:00+08:00                 2           13.780000       4.806500      1.866951
14  checkoutservice-0  container.node-5.container_cpu_cfs_throttled_s... 2022-03-21 18:06:00+08:00                 2            0.450242       0.173462      1.595621
15     cartservice2-0  container.node-6.container_network_receive_MB.... 2022-03-21 18:07:00+08:00                 2            0.457124       0.216149      1.114858
16             node-1                               system.cpu.pct_usage 2022-03-21 18:02:00+08:00                 2           15.760000       7.480000      1.106952
17      cartservice-0  container.node-5.container_cpu_cfs_throttled_s... 2022-03-21 18:03:00+08:00                 2            0.052455       0.028850      0.818182
18      cartservice-1  container.node-5.container_cpu_cfs_throttled_s... 2022-03-21 18:21:00+08:00                 2            0.052502       0.028876      0.818182
19        adservice-2  container.node-5.container_cpu_cfs_throttled_s... 2022-03-21 18:09:00+08:00                 2            0.004908       0.002699      0.818182, 402, 4, 16)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cached metric_faults_real_severity
if 'metric_faults_real_severity' not in globals():
    metric_faults_real_severity = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'duration_minutes','max_value_in_fault','p95_threshold','severity'
    ])

# Ensure fault_start_timestamp is datetime and timezone-aware (Asia/Shanghai)
if 'fault_start_timestamp' in metric_faults_real_severity.columns and not metric_faults_real_severity.empty:
    metric_faults_real_severity['fault_start_timestamp'] = pd.to_datetime(metric_faults_real_severity['fault_start_timestamp'])
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    metric_faults_real_severity['fault_start_timestamp'] = metric_faults_real_severity['fault_start_timestamp'].apply(_ensure_tz)

# Target timestamp
target_ts = pd.Timestamp('2022-03-21 18:14:00', tz=tz)

# Filter rows where fault_start_timestamp equals target
faults_at_18_14 = metric_faults_real_severity[
    metric_faults_real_severity['fault_start_timestamp'] == target_ts
].copy()

# Prepare compact display (limit to 50 rows)
display_cols = ['cmdb_id', 'kpi_name', 'fault_start_timestamp', 'duration_minutes', 'max_value_in_fault', 'p95_threshold', 'severity']
faults_at_18_14_head = faults_at_18_14[display_cols].head(50)

# Additional summaries
unique_cmdb_count_at_18_14 = int(faults_at_18_14['cmdb_id'].nunique())
top10_cmdb_by_count = faults_at_18_14['cmdb_id'].value_counts().head(10).rename_axis('cmdb_id').reset_index(name='count')

# Return compact results (variables shown by IPython)
faults_at_18_14_head, unique_cmdb_count_at_18_14, top10_cmdb_by_count

```
Out[10]:
```
```python
# Build a concise plain-English summary for faults starting at 2022-03-21 18:14:00+08:00
# Uses cached variables in the kernel: faults_at_18_14 (full DF), faults_at_18_14_head, unique_cmdb_count_at_18_14, top10_cmdb_by_count

try:
    total_faults = int(faults_at_18_14.shape[0])
except Exception:
    total_faults = 0

try:
    unique_cmdbs = int(unique_cmdb_count_at_18_14)
except Exception:
    unique_cmdbs = int(faults_at_18_14['cmdb_id'].nunique()) if 'faults_at_18_14' in globals() else 0

# Build top cmdb list
top_list = []
if 'top10_cmdb_by_count' in globals() and not top10_cmdb_by_count.empty:
    for _, r in top10_cmdb_by_count.iterrows():
        top_list.append(f"{r['cmdb_id']}({int(r['count'])})")
else:
    top_counts = faults_at_18_14['cmdb_id'].value_counts().head(10)
    top_list = [f"{idx}({int(cnt)})" for idx, cnt in top_counts.items()]

summary = (
    f"Faults at 2022-03-21 18:14:00+08:00 — summary:\n"
    f"- Total faults starting at 18:14:00: {total_faults}\n"
    f"- Unique cmdb_id affected: {unique_cmdbs}\n"
    f"- Top cmdb_id by fault count (top 10): {', '.join(top_list)}\n\n"
    f"- Observations from the sample faults:\n"
    f"  * The highest-count hosts are checkoutservice-1 and paymentservice2-0 (each with multiple faults).\n"
    f"  * Many faults are short (duration = 2 minutes) and involve mesh/istio metrics for checkoutservice (e.g., jaeger-collector and frontend-checkout istio metrics) with very large max values and high computed severity.\n"
    f"  * Paymentservice2-0 entries relate to container disk reads and memory-failure counters (inspect fs read MB and memory failure metrics on node-6/pods).\n"
    f"  * Examples include very large mesh counter values for checkoutservice (max values up to ~28,825) and elevated jaeger/istio metrics indicating heavy tracing/mesh activity at 18:14.\n\n"
    f"- Recommended next steps:\n"
    f"  1) Investigate checkoutservice request/trace flows and jaeger/collector activity around 18:14 (mesh destination/source metrics show strong breaches).\n"
    f"  2) Inspect paymentservice2-0 pod/node I/O and memory-failure counters on node-6 for correlated resource issues.\n"
    f"  3) Correlate these faults with application logs and trace spans for requests at ~18:14 to identify root cause.\n\n"
    f"(Full filtered table stored in kernel variable 'faults_at_18_14'.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name     fault_start_timestamp  duration_minutes  max_value_in_fault  p95_threshold      severity
0    checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i... 2022-03-21 18:14:00+08:00                 2          126.047100       0.002467  51081.917934
1    checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2          475.850000       6.200000     75.750000
3    checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2        16092.500000     314.000000     50.250000
4    checkoutservice-1  mesh.destination.frontend.checkoutservice.isti... 2022-03-21 18:14:00+08:00                 2        28825.000000     580.000000     48.698276
5      checkoutservice                                       app.grpc.mrt 2022-03-21 18:14:00+08:00                 2           21.206364       1.073600     18.752575
7    checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i... 2022-03-21 18:14:00+08:00                 2          172.611050      22.203750      6.773959
9    paymentservice2-0    container.node-6.container_fs_reads_MB./dev/vda 2022-03-21 18:14:00+08:00                 2            0.111979       0.030794      2.636364
10   paymentservice2-0       container.node-6.container_fs_reads./dev/vda 2022-03-21 18:14:00+08:00                 2            5.333333       1.466667      2.636364
11   paymentservice2-0  container.node-6.container_memory_failures.con... 2022-03-21 18:14:00+08:00                 2            0.666667       0.183333      2.636364
12   paymentservice2-0  container.node-6.container_memory_failures.hie... 2022-03-21 18:14:00+08:00                 2            0.666667       0.183333      2.636364
21    paymentservice-1      container.node-5.container_cpu_system_seconds 2022-03-21 18:14:00+08:00                 2            0.005000       0.002750      0.818182
106         frontend-0  container.node-5.container_cpu_cfs_throttled_s... 2022-03-21 18:14:00+08:00                 2            0.003797       0.003499      0.085409
110      cartservice-2      container.node-5.container_cpu_system_seconds 2022-03-21 18:14:00+08:00                 2            0.080000       0.075000      0.066667, 7,              cmdb_id  count
0  checkoutservice-1      4
1  paymentservice2-0      4
2  checkoutservice-0      1
3    checkoutservice      1
4   paymentservice-1      1
5         frontend-0      1
6      cartservice-2      1)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cached df_trace
# Ensure timezone-aware timestamp column 'ts' exists
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if df_trace['ts'].dt.tz is None:
        df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_trace['ts'] = df_trace['ts'].dt.tz_convert(tz)

# Target single minute: 2022-03-21 18:14:00+08:00 (include whole minute)
target_min = pd.Timestamp('2022-03-21 18:14:00', tz=tz)
# Filter rows where floor to minute equals target_min
trace_18_14 = df_trace[df_trace['ts'].dt.floor('T') == target_min].copy()

# Select trace_name values containing any of the keywords checkout, payment, email (case-insensitive)
pattern = r'(checkout|payment|email)'
trace_18_14 = trace_18_14[trace_18_14['trace_name'].str.contains(pattern, case=False, na=False)].copy()

# Aggregate by (cmdb_id, trace_name)
trace_agg = (
    trace_18_14.groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        earliest_timestamp=('ts', 'min')
    )
)

# Sort by count desc and keep top 20
trace_agg_top20 = trace_agg.sort_values(by='count', ascending=False).head(20)

# Return compact results: aggregated top20 and store full minute DataFrame in 'trace_18_14'
trace_agg_top20, trace_18_14.shape

```
Out[11]:
```
- I filtered trace records to 2022-03-21 18:14:00+08:00 and kept only trace_names containing checkout, payment, or email (case-insensitive). The filtered minute dataframe (trace_18_14) contains 312 rows.

- Aggregation by (cmdb_id, trace_name) shows most pairs have a single sample in that minute (counts = 1). In the top-20 aggregates:
  - Many metrics are duration_p95 or error_rate with mean/max = 0.0.
  - Non-zero row_count values appear for some services: e.g., shippingservice2-0 trace.from_checkoutservice2-0.row_count = 12, and several cartservice/* trace.from_checkoutservice-*.row_count entries = 2–3.
  - All earliest timestamps are at 2022-03-21 18:14:00+08:00 (the selected minute).

- Interpretation / next steps:
  - The minute shows broad activity across checkout/payment/email-related traces, but most duration/error metrics are zero while a few row_count metrics indicate actual request counts (notably shippingservice2-0 and cartservice instances).
  - Investigate the non-zero row_count traces (shippingservice2-0, cartservice, and any checkoutservice traces) and correlate them with the faults observed at 18:14 (see faults_at_18_14) to find whether these request spikes align with the high-severity metric faults.

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id                                  trace_name  count  mean_value  max_value        earliest_timestamp
311  shippingservice2-0     trace.from_checkoutservice2-0.row_count      1        12.0       12.0 2022-03-21 18:14:00+08:00
0         cartservice-0  trace.from_checkoutservice-0.duration_mean      1         0.0        0.0 2022-03-21 18:14:00+08:00
1         cartservice-0   trace.from_checkoutservice-0.duration_p95      1         0.0        0.0 2022-03-21 18:14:00+08:00
2         cartservice-0     trace.from_checkoutservice-0.error_rate      1         0.0        0.0 2022-03-21 18:14:00+08:00
3         cartservice-0      trace.from_checkoutservice-0.row_count      1         3.0        3.0 2022-03-21 18:14:00+08:00
4         cartservice-0  trace.from_checkoutservice-1.duration_mean      1         0.0        0.0 2022-03-21 18:14:00+08:00
5         cartservice-0   trace.from_checkoutservice-1.duration_p95      1         0.0        0.0 2022-03-21 18:14:00+08:00
6         cartservice-0     trace.from_checkoutservice-1.error_rate      1         0.0        0.0 2022-03-21 18:14:00+08:00
7         cartservice-0      trace.from_checkoutservice-1.row_count      1         2.0        2.0 2022-03-21 18:14:00+08:00
8         cartservice-0  trace.from_checkoutservice-2.duration_mean      1         0.0        0.0 2022-03-21 18:14:00+08:00
9         cartservice-0   trace.from_checkoutservice-2.duration_p95      1         0.0        0.0 2022-03-21 18:14:00+08:00
10        cartservice-0     trace.from_checkoutservice-2.error_rate      1         0.0        0.0 2022-03-21 18:14:00+08:00
11        cartservice-0      trace.from_checkoutservice-2.row_count      1         2.0        2.0 2022-03-21 18:14:00+08:00
12        cartservice-1  trace.from_checkoutservice-0.duration_mean      1         0.0        0.0 2022-03-21 18:14:00+08:00
13        cartservice-1   trace.from_checkoutservice-0.duration_p95      1         0.0        0.0 2022-03-21 18:14:00+08:00
14        cartservice-1     trace.from_checkoutservice-0.error_rate      1         0.0        0.0 2022-03-21 18:14:00+08:00
15        cartservice-1      trace.from_checkoutservice-0.row_count      1         3.0        3.0 2022-03-21 18:14:00+08:00
16        cartservice-1  trace.from_checkoutservice-1.duration_mean      1         0.0        0.0 2022-03-21 18:14:00+08:00
17        cartservice-1   trace.from_checkoutservice-1.duration_p95      1         0.0        0.0 2022-03-21 18:14:00+08:00
18        cartservice-1     trace.from_checkoutservice-1.error_rate      1         0.0        0.0 2022-03-21 18:14:00+08:00, (312, 5))```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_18_14 exists
if 'trace_18_14' not in globals():
    trace_18_14 = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','ts'])

# Ensure 'ts' exists and is timezone-aware Asia/Shanghai
if 'ts' not in trace_18_14.columns:
    if 'timestamp' in trace_18_14.columns:
        trace_18_14['ts'] = pd.to_datetime(trace_18_14['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace_18_14['ts'] = pd.NaT
else:
    # convert/ensure tz-aware
    trace_18_14['ts'] = pd.to_datetime(trace_18_14['ts'])
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    trace_18_14['ts'] = trace_18_14['ts'].apply(_ensure_tz)

# 1) Filter rows by trace_name keywords and value > 0
pattern = r'(row_count|duration_p95|duration_mean|error_rate)'
mask_name = trace_18_14['trace_name'].astype(str).str.contains(pattern, case=False, na=False)
mask_value = pd.to_numeric(trace_18_14['value'], errors='coerce') > 0
trace_18_14_nonzero = trace_18_14[mask_name & mask_value].copy()

# 2) Aggregate by (cmdb_id, trace_name)
if not trace_18_14_nonzero.empty:
    trace_agg = (
        trace_18_14_nonzero
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            count=('value', 'count'),
            mean_value=('value', 'mean'),
            max_value=('value', 'max'),
            earliest_timestamp=('ts', 'min')
        )
    )
    # Ensure earliest_timestamp is timezone-aware Asia/Shanghai
    trace_agg['earliest_timestamp'] = pd.to_datetime(trace_agg['earliest_timestamp']).apply(
        lambda x: x.tz_localize(tz) if (pd.notna(x) and x.tzinfo is None) else (x.tz_convert(tz) if pd.notna(x) else x)
    )
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count','mean_value','max_value','earliest_timestamp'])

# 3) Top 20 by max_value desc
trace_agg_top20 = trace_agg.sort_values(by='max_value', ascending=False).head(20)

# 4) Number of unique cmdb_id in the filtered set
unique_cmdb_count = int(trace_18_14_nonzero['cmdb_id'].nunique())

# Return compact results: aggregated top20, unique cmdb count, and keep the pre-aggregated DF cached
trace_agg_top20, unique_cmdb_count, trace_18_14_nonzero.shape

```
Out[12]:
```
```python
# Compose a plain-English summary using cached trace_18_14_nonzero and trace_agg_top20 in the kernel
try:
    n_rows = int(trace_18_14_nonzero.shape[0])
except Exception:
    n_rows = 0

try:
    unique_cmdb = int(unique_cmdb_count)
except Exception:
    unique_cmdb = int(trace_18_14_nonzero['cmdb_id'].nunique()) if 'trace_18_14_nonzero' in globals() and not trace_18_14_nonzero.empty else 0

# Prepare top examples from aggregated results if available
top_examples = []
if 'trace_agg_top20' in globals() and not trace_agg_top20.empty:
    for _, r in trace_agg_top20.head(6).iterrows():
        top_examples.append(f"{r['cmdb_id']} | {r['trace_name']} | max={r['max_value']:.0f} | count={int(r['count'])}")
else:
    top_examples = []

summary = (
    f"Trace-minute (2022-03-21 18:14:00+08:00) summary (filtered to row_count/duration_p95/duration_mean/error_rate > 0):\n"
    f"- Total filtered rows: {n_rows}\n"
    f"- Unique cmdb_id: {unique_cmdb}\n"
    f"- Top aggregated pairs by max value (sample top 6):\n"
)

if top_examples:
    for ex in top_examples:
        summary += f"  - {ex}\n"
else:
    summary += "  - (no non-zero matching traces found)\n"

summary += (
    "- Observations:\n"
    "  * Most aggregates are single-sample entries (count = 1) for this minute; mean == max for many rows.\n"
    "  * Noticeable non-zero row_count spikes include currencyservice2-0 (max=18), several services with max=12 (cartservice2-0, shippingservice2-0, productcatalogservice2-0), and frontend->checkout row_count entries (e.g., frontend2-0 max=7).\n"
    "  * All earliest timestamps are within the selected minute (2022-03-21 18:14:00+08:00).\n"
    "- Recommendation: investigate the listed services (currencyservice2-0, cartservice2-0, shippingservice2-0, productcatalogservice2-0, and checkout/payment related traces) and correlate with the faults at 18:14, application logs, and traces to find root cause of the request/activity spike.\n"
    "\n(Full filtered minute DataFrame is stored in kernel variable 'trace_18_14_nonzero'; aggregated results are in 'trace_agg_top20'.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                trace_name  count  mean_value  max_value        earliest_timestamp
92         currencyservice2-0   trace.from_checkoutservice2-0.row_count      1   18.000000  18.000000 2022-03-21 18:14:00+08:00
11             cartservice2-0   trace.from_checkoutservice2-0.row_count      1   12.000000  12.000000 2022-03-21 18:14:00+08:00
215        shippingservice2-0   trace.from_checkoutservice2-0.row_count      1   12.000000  12.000000 2022-03-21 18:14:00+08:00
185  productcatalogservice2-0   trace.from_checkoutservice2-0.row_count      1   12.000000  12.000000 2022-03-21 18:14:00+08:00
131               frontend2-0     trace.to_checkoutservice2-0.row_count      1    7.000000   7.000000 2022-03-21 18:14:00+08:00
62         checkoutservice2-0      trace.to_paymentservice2-0.row_count      1    6.000000   6.000000 2022-03-21 18:14:00+08:00
59         checkoutservice2-0        trace.to_emailservice2-0.row_count      1    6.000000   6.000000 2022-03-21 18:14:00+08:00
116           emailservice2-0   trace.from_checkoutservice2-0.row_count      1    6.000000   6.000000 2022-03-21 18:14:00+08:00
158         paymentservice2-0   trace.from_checkoutservice2-0.row_count      1    6.000000   6.000000 2022-03-21 18:14:00+08:00
83          currencyservice-2    trace.from_checkoutservice-0.row_count      1    4.000000   4.000000 2022-03-21 18:14:00+08:00
65          currencyservice-0    trace.from_checkoutservice-0.row_count      1    4.000000   4.000000 2022-03-21 18:14:00+08:00
74          currencyservice-1    trace.from_checkoutservice-0.row_count      1    4.000000   4.000000 2022-03-21 18:14:00+08:00
130               frontend2-0  trace.to_checkoutservice2-0.duration_p95      1    3.301002   3.301002 2022-03-21 18:14:00+08:00
71          currencyservice-0    trace.from_checkoutservice-2.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
176   productcatalogservice-2    trace.from_checkoutservice-0.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
170   productcatalogservice-1    trace.from_checkoutservice-0.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
119                frontend-0      trace.to_checkoutservice-0.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
206         shippingservice-2    trace.from_checkoutservice-0.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
125                frontend-0      trace.to_checkoutservice-2.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00
89          currencyservice-2    trace.from_checkoutservice-2.row_count      1    3.000000   3.000000 2022-03-21 18:14:00+08:00, 31, (216, 5))```
```

In [ ]:
# In[13]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing faults data: prefer metric_faults_real_severity, then metric_faults_severity, then metric_faults
if 'metric_faults_real_severity' in globals():
    src_faults = metric_faults_real_severity
elif 'metric_faults_severity' in globals():
    src_faults = metric_faults_severity
elif 'metric_faults' in globals():
    src_faults = metric_faults
else:
    src_faults = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'duration_minutes','max_value_in_fault','p95_threshold','severity'
    ])

# Ensure timestamp column exists and is timezone-aware Asia/Shanghai
if 'fault_start_timestamp' in src_faults.columns and not src_faults.empty:
    src_faults['fault_start_timestamp'] = pd.to_datetime(src_faults['fault_start_timestamp'])
    def _ensure_tz(ts):
        if pd.isna(ts):
            return ts
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    src_faults['fault_start_timestamp'] = src_faults['fault_start_timestamp'].apply(_ensure_tz)

# Target timestamp
target_ts = pd.Timestamp('2022-03-21 18:29:00', tz=tz)

# Filter faults that start exactly at target timestamp
faults_at_18_29 = src_faults[src_faults['fault_start_timestamp'] == target_ts].copy()

# Prepare compact view (limit 20 rows) with requested columns
display_cols = ['cmdb_id', 'kpi_name', 'fault_start_timestamp', 'duration_minutes', 'max_value_in_fault', 'p95_threshold', 'severity']
faults_at_18_29_head = faults_at_18_29[display_cols].head(20)

# Compute number of unique cmdb_id in this filtered set
unique_cmdb_count_18_29 = int(faults_at_18_29['cmdb_id'].nunique())

# Top 10 cmdb_id by fault count in this filtered set
top10_cmdb_by_count_18_29 = (
    faults_at_18_29['cmdb_id'].value_counts().head(10)
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# Store filtered result in kernel variable as requested (already assigned)
# Return compact outputs
faults_at_18_29_head, unique_cmdb_count_18_29, top10_cmdb_by_count_18_29

```
Out[13]:
```
```python
# Plain-English summary of the 18:29 filter result (based on the previously executed filter)
summary = (
    "Result summary:\n"
    "- No metric faults start exactly at 2022-03-21 18:29:00+08:00 in the cached faults table; the filtered table is empty.\n"
    "- Unique cmdb_id in this filtered set: 0\n"
    "- Top-10 cmdb_id by fault count: none (no rows)\n\n"
    "Note: other timestamps (for example, 2022-03-21 18:14:00+08:00 and ~18:02:00+08:00) did contain faults in earlier analyses, so investigate those times if you are tracing incident activity."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, fault_start_timestamp, duration_minutes, max_value_in_fault, p95_threshold, severity]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [])```
```